# Export D3 EC percentage by guide

This notebook recreates:

`d3_EC_percent_by_guide_with_individual_NT_excl_Hippo.csv`

from the integrated CC-Perturb-seq AnnData object on Sherlock.

Definitions used:

- D3 cells: `adata_int.obs["day"] == "d3"`
- EC cells: `adata_int.obs["cell_type"] == "artEC"`
- targeting-guide rows: cells carrying the query guide **and no NT/SAFE guide**
- individual NT-guide rows: cells carrying that control guide, excluding cells carrying any Hippo guide
- pooled NT control row: cells carrying any NT/SAFE control guide, excluding cells carrying any Hippo guide
- Hippo targets excluded from controls: `NF2`, `LATS2`, `SAV1`, `TAOK1`, `AMOTL2`

Thus, each query-guide group is mutually exclusive with the NT/SAFE-guide groups.

The output schema matches the previously used CSV.


## 1. Configuration

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse


# ============================================================
# PATHS
# ============================================================

ADATA_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
    "2.preprocessing_tonys_pipeline/3.integrated_days/"
    "3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad"
)

WORKDIR = Path(
    "/oak/stanford/groups/engreitz/Users/tri/CodeBase/"
    "programDE/notebooks/per_day_umaps"
)

# The guide metadata file previously used for this project.
# The notebook also searches WORKDIR/data and the current directory.
GUIDE_METADATA_PATH = WORKDIR / "300genes_guide_metadata_v43.tsv"

OUTPUT_PATH = (
    WORKDIR
    / "d3_EC_percent_by_guide_with_individual_NT_excl_Hippo.csv"
)

MAPPING_QC_PATH = (
    WORKDIR
    / "d3_EC_percent_by_guide_with_individual_NT_excl_Hippo_guide_mapping_QC.csv"
)

QUERY_EXCLUSION_QC_PATH = (
    WORKDIR
    / "d3_EC_percent_by_guide_with_individual_NT_excl_Hippo_query_exclusion_QC.csv"
)

WORKDIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# DATA KEYS AND BIOLOGICAL DEFINITIONS
# ============================================================

DAY_COL = "day"
CELLTYPE_COL = "cell_type"

GUIDE_NAMES_KEY = "guide_names"
GUIDE_ASSIGNMENT_KEY = "guide_assignment"

D3_LABEL = "d3"

# This reproduces the prior file: only artEC is counted as D3 EC.
D3_EC_LABELS = {"artEC"}

HIPPO_TARGETS = {
    "NF2",
    "LATS2",
    "SAV1",
    "TAOK1",
    "AMOTL2",
}

# Use backed mode so the expression matrix is not loaded into memory.
# The guide-assignment matrix is subset to D3 below.
USE_BACKED_MODE = True

SAVE_MAPPING_QC = True


# Expected values for the current integrated object.
# These generate warnings, not hard failures, if the source object changes.
EXPECTED_TOTAL_GUIDES = 1922
EXPECTED_CONTROL_GUIDES = 120
EXPECTED_TARGETING_GUIDES = 1802
EXPECTED_TARGETS = 298

print("AnnData:", ADATA_PATH)
print("Guide metadata:", GUIDE_METADATA_PATH)
print("Output:", OUTPUT_PATH)


## 2. Helper functions

In [ ]:

# ============================================================
# HELPERS
# ============================================================

# Fallback aliases from old guide-name symbols to current GENCODE-v43 symbols.
# These are used only when a targeting guide is absent from the metadata file.
GENE_SYMBOL_ALIASES = {
    "ATP5F1": "ATP5PB",
    "C16orf93": "CFAP119",
    "C2orf53": "PRR30",
    "C7orf26": "INTS15",
    "CCDC101": "SGF29",
    "EPT1": "SELENOI",
    "ERO1L": "ERO1A",
    "FAM208A": "TASOR",
    "GNB2L1": "RACK1",
    "KIAA1432": "RIC1",
    "KIAA1551": "RESF1",
    "LUZP6": "MTPN",
    "MESDC1": "TLNRD1",
    "SRPR": "SRPRA",
    "SSFA2": "ITPRID2",
    "T": "TBXT",
    "TCEB3": "ELOA",
    "VWA9": "INTS14",
    "WAPAL": "WAPL",
    "WDR61": "SKIC8",
}


def is_control_guide(guide_name):
    """
    Identify the two control-guide classes in this library:
      - SAFE_TARGETING_*
      - non-targeting_*
    """
    s = str(guide_name).strip().lower()

    return (
        s.startswith("safe_targeting")
        or s.startswith("safe-targeting")
        or s.startswith("non-targeting")
        or s.startswith("non_targeting")
        or s.startswith("nontargeting")
        or s.startswith("ntc")
    )


def parse_target_from_guide_name(guide_name):
    """
    Fallback target parser for targeting guides.

    Examples:
      ACAA1_+_38178488.23-P1P2 -> ACAA1
      WDR61_SGTN_13             -> WDR61 -> SKIC8
    """
    prefix = str(guide_name).split("_", 1)[0].strip()
    return GENE_SYMBOL_ALIASES.get(prefix, prefix)


def find_guide_metadata_path(explicit_path):
    candidates = [
        Path(explicit_path),
        WORKDIR / "300genes_guide_metadata_v43.tsv",
        WORKDIR / "data" / "300genes_guide_metadata_v43.tsv",
        Path.cwd() / "300genes_guide_metadata_v43.tsv",
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find 300genes_guide_metadata_v43.tsv.\n"
        "Checked:\n  "
        + "\n  ".join(str(p) for p in candidates)
        + "\n\nUpdate GUIDE_METADATA_PATH in the configuration cell."
    )


def read_guide_metadata(path):
    """
    Read guide metadata and return a guide_id -> target mapping.

    Expected columns:
      guide_id
      putative_target_genes
    """
    path = Path(path)

    if path.suffix.lower() in {".tsv", ".txt"}:
        meta = pd.read_csv(path, sep="\t")
    else:
        meta = pd.read_csv(path)

    meta.columns = [str(c).strip() for c in meta.columns]

    guide_col_candidates = [
        "guide_id",
        "guide",
        "guide_name",
        "sgRNA",
        "sgrna",
        "gRNA",
    ]
    target_col_candidates = [
        "putative_target_genes",
        "target",
        "target_gene",
        "gene",
        "gene_symbol",
    ]

    guide_col = next(
        (c for c in guide_col_candidates if c in meta.columns),
        None,
    )
    target_col = next(
        (c for c in target_col_candidates if c in meta.columns),
        None,
    )

    if guide_col is None or target_col is None:
        raise ValueError(
            "Could not identify guide/target columns in guide metadata.\n"
            f"Available columns: {meta.columns.tolist()}"
        )

    mapping_df = meta[[guide_col, target_col]].copy()
    mapping_df.columns = ["guide_id", "target"]
    mapping_df["guide_id"] = mapping_df["guide_id"].astype(str).str.strip()
    mapping_df["target"] = mapping_df["target"].astype("string").str.strip()

    # Keep one mapping per guide and warn if conflicting targets occur.
    conflicts = (
        mapping_df.dropna(subset=["target"])
        .groupby("guide_id")["target"]
        .nunique()
    )
    conflicts = conflicts[conflicts > 1]

    if len(conflicts) > 0:
        print(
            "[warning] Conflicting target annotations were found for",
            len(conflicts),
            "guides. Keeping the first annotation."
        )

    mapping_df = mapping_df.drop_duplicates("guide_id", keep="first")

    return mapping_df


def get_d3_guide_assignment(adata, d3_indices):
    """
    Subset the cell x guide assignment matrix to D3.

    Works with pandas DataFrame, dense NumPy arrays, and scipy sparse matrices.
    """
    ga = adata.obsm[GUIDE_ASSIGNMENT_KEY]

    if isinstance(ga, pd.DataFrame):
        out = ga.iloc[d3_indices].to_numpy()

    elif sparse.issparse(ga):
        out = ga[d3_indices, :].tocsr()

    else:
        # Backed arrays and dense NumPy arrays generally support tuple indexing.
        try:
            out = ga[d3_indices, :]
        except Exception:
            out = np.asarray(ga)[d3_indices, :]

        out = np.asarray(out)

    # Treat any positive guide-assignment value as assigned.
    if sparse.issparse(out):
        out = out.astype(bool)
    else:
        out = np.asarray(out) > 0

    return out


def column_sums(matrix, row_mask=None):
    """
    Sum guide-carrier cells by guide column.
    """
    sub = matrix if row_mask is None else matrix[row_mask, :]

    values = np.asarray(sub.sum(axis=0)).ravel()

    return values.astype(int)


def row_any(matrix, column_mask):
    """
    Return whether each cell carries at least one guide in column_mask.
    """
    cols = np.flatnonzero(np.asarray(column_mask, dtype=bool))

    if len(cols) == 0:
        return np.zeros(matrix.shape[0], dtype=bool)

    sub = matrix[:, cols]

    if sparse.issparse(sub):
        return np.asarray(sub.getnnz(axis=1)).ravel() > 0

    return np.any(np.asarray(sub), axis=1)


def safe_percent(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)

    result = np.full(denominator.shape, np.nan, dtype=float)

    valid = denominator > 0
    result[valid] = 100.0 * numerator[valid] / denominator[valid]

    return result


def warn_if_different(name, observed, expected):
    if observed != expected:
        print(
            f"[warning] {name}: observed {observed}, expected {expected}. "
            "The source object or guide metadata may differ from the prior export."
        )
    else:
        print(f"[OK] {name}: {observed}")


## 3. Load the integrated AnnData and guide metadata

In [ ]:

# ============================================================
# LOAD ANNDATA AND GUIDE METADATA
# ============================================================

if not ADATA_PATH.exists():
    raise FileNotFoundError(f"AnnData file not found:\n{ADATA_PATH}")

metadata_path = find_guide_metadata_path(GUIDE_METADATA_PATH)
guide_metadata = read_guide_metadata(metadata_path)

print("Using guide metadata:", metadata_path)
print("Guide metadata rows:", len(guide_metadata))

backed_mode = "r" if USE_BACKED_MODE else None
adata_int = sc.read_h5ad(ADATA_PATH, backed=backed_mode)

required_obs = [DAY_COL, CELLTYPE_COL]
missing_obs = [c for c in required_obs if c not in adata_int.obs.columns]

if missing_obs:
    raise KeyError(
        f"Missing required adata.obs columns: {missing_obs}"
    )

if GUIDE_NAMES_KEY not in adata_int.uns:
    raise KeyError(
        f"adata_int.uns['{GUIDE_NAMES_KEY}'] was not found."
    )

if GUIDE_ASSIGNMENT_KEY not in adata_int.obsm:
    raise KeyError(
        f"adata_int.obsm['{GUIDE_ASSIGNMENT_KEY}'] was not found."
    )

guide_names = np.asarray(
    adata_int.uns[GUIDE_NAMES_KEY]
).astype(str)

obs = adata_int.obs[[DAY_COL, CELLTYPE_COL]].copy()

day_values = obs[DAY_COL].astype(str).str.lower()
celltype_values = obs[CELLTYPE_COL].astype(str)

d3_mask_all = day_values.eq(D3_LABEL.lower()).to_numpy()
d3_indices = np.flatnonzero(d3_mask_all)

d3_celltypes = celltype_values.iloc[d3_indices].to_numpy()
d3_ec_mask = np.isin(d3_celltypes, list(D3_EC_LABELS))

guide_assignment_d3 = get_d3_guide_assignment(
    adata_int,
    d3_indices,
)

if guide_assignment_d3.shape[1] != len(guide_names):
    raise ValueError(
        "Guide-assignment columns do not match guide_names length:\n"
        f"guide_assignment columns = {guide_assignment_d3.shape[1]}\n"
        f"guide_names = {len(guide_names)}"
    )

print("Total cells:", adata_int.n_obs)
print("D3 cells:", len(d3_indices))
print("D3 artEC cells:", int(d3_ec_mask.sum()))
print("Guide-assignment D3 shape:", guide_assignment_d3.shape)


## 4. Build the guide-to-target map

In [ ]:

# ============================================================
# BUILD GUIDE -> TARGET MAPPING
# ============================================================

metadata_target_map = (
    guide_metadata
    .set_index("guide_id")["target"]
    .to_dict()
)

control_mask_guides = np.array(
    [is_control_guide(g) for g in guide_names],
    dtype=bool,
)

target_rows = []
unmapped_targeting_guides = []

for guide_index, guide_name in enumerate(guide_names):
    if control_mask_guides[guide_index]:
        target = pd.NA
        source = "control_guide_name_pattern"

    else:
        target = metadata_target_map.get(guide_name, pd.NA)

        if pd.isna(target) or str(target).strip() == "":
            target = parse_target_from_guide_name(guide_name)
            source = "guide_name_fallback"
            unmapped_targeting_guides.append(guide_name)
        else:
            target = str(target).strip()
            source = "300genes_guide_metadata_v43.tsv"

    target_rows.append({
        "guide_index": guide_index,
        "guide_id": guide_name,
        "guide_name": guide_name,
        "is_control_guide": bool(control_mask_guides[guide_index]),
        "target": target,
        "target_mapping_source": source,
    })

guide_map = pd.DataFrame(target_rows)

if len(unmapped_targeting_guides) > 0:
    print(
        "[warning]",
        len(unmapped_targeting_guides),
        "targeting guides were absent from guide metadata and used guide-name fallback."
    )
else:
    print("[OK] All targeting guides mapped through the metadata file.")

targeting_guide_map = guide_map.loc[
    ~guide_map["is_control_guide"]
].copy()

control_guide_map = guide_map.loc[
    guide_map["is_control_guide"]
].copy()

n_targets = targeting_guide_map["target"].nunique(dropna=True)

warn_if_different(
    "total guides",
    len(guide_map),
    EXPECTED_TOTAL_GUIDES,
)
warn_if_different(
    "control guides",
    len(control_guide_map),
    EXPECTED_CONTROL_GUIDES,
)
warn_if_different(
    "targeting guides",
    len(targeting_guide_map),
    EXPECTED_TARGETING_GUIDES,
)
warn_if_different(
    "unique targets",
    n_targets,
    EXPECTED_TARGETS,
)

mapped_hippo_targets = set(
    targeting_guide_map.loc[
        targeting_guide_map["target"].isin(HIPPO_TARGETS),
        "target",
    ]
)

missing_hippo_targets = HIPPO_TARGETS - mapped_hippo_targets

if missing_hippo_targets:
    print(
        "[warning] Hippo targets not found in guide map:",
        sorted(missing_hippo_targets),
    )
else:
    print("[OK] All Hippo targets found:", sorted(HIPPO_TARGETS))

if SAVE_MAPPING_QC:
    guide_map.to_csv(MAPPING_QC_PATH, index=False)
    print("Saved mapping QC:", MAPPING_QC_PATH)

guide_map.head()


## 5. Compute and export D3 EC percentages

In [ ]:

# ============================================================
# COMPUTE D3 EC PERCENTAGE BY GUIDE
# ============================================================

guide_targets = guide_map["target"].astype("string")

targeting_cols = ~guide_map["is_control_guide"].to_numpy()
control_cols = guide_map["is_control_guide"].to_numpy()

hippo_guide_cols = (
    guide_targets.isin(HIPPO_TARGETS).fillna(False).to_numpy()
)

# ------------------------------------------------------------------
# Determine whether each D3 cell carries any NT/SAFE guide.
#
# Query-guide groups are defined only among cells with no control guide,
# so query-guide carriers and NT/SAFE-guide carriers are mutually exclusive.
# ------------------------------------------------------------------

raw_control_carrier_mask_d3 = row_any(
    guide_assignment_d3,
    control_cols,
)

eligible_query_cell_mask = ~raw_control_carrier_mask_d3

assert not np.any(
    raw_control_carrier_mask_d3[eligible_query_cell_mask]
)


# ------------------------------------------------------------------
# Targeting/query guides
#
# For every query guide:
#   include cells carrying that guide
#   exclude cells carrying any NT/SAFE guide
#
# Other targeting-guide co-carriers are retained.
# ------------------------------------------------------------------

targeting_matrix = guide_assignment_d3[:, targeting_cols]

# Counts before NT/SAFE exclusion, retained only for QC.
targeting_total_raw = column_sums(
    targeting_matrix
)

targeting_ec_raw = column_sums(
    targeting_matrix,
    row_mask=d3_ec_mask,
)

# Counts used in the output.
targeting_total = column_sums(
    targeting_matrix,
    row_mask=eligible_query_cell_mask,
)

targeting_ec = column_sums(
    targeting_matrix,
    row_mask=eligible_query_cell_mask & d3_ec_mask,
)

targeting_nonec = targeting_total - targeting_ec
targeting_pct = safe_percent(
    targeting_ec,
    targeting_total,
)

targeting_out = guide_map.loc[
    targeting_cols,
    ["guide_id", "guide_name", "guide_index", "target"],
].copy()

targeting_out.insert(
    0,
    "row_type",
    "targeting_guide",
)

targeting_out["control_definition"] = pd.NA
targeting_out["exclusion_rule"] = "exclude_NT_SAFE_cocarriers"
targeting_out["n_D3_cells"] = targeting_total
targeting_out["n_D3_EC_cells"] = targeting_ec
targeting_out["n_D3_nonEC_cells"] = targeting_nonec
targeting_out["pct_EC_D3"] = targeting_pct


# ------------------------------------------------------------------
# Save query-guide exclusion QC table.
# ------------------------------------------------------------------

query_exclusion_qc = targeting_out[
    ["guide_id", "guide_name", "guide_index", "target"]
].copy()

query_exclusion_qc["n_D3_query_carriers_before_NT_SAFE_exclusion"] = (
    targeting_total_raw
)

query_exclusion_qc["n_D3_query_carriers_after_NT_SAFE_exclusion"] = (
    targeting_total
)

query_exclusion_qc["n_D3_query_carriers_removed_for_NT_SAFE_cocarriage"] = (
    targeting_total_raw - targeting_total
)

query_exclusion_qc["n_D3_EC_before_NT_SAFE_exclusion"] = (
    targeting_ec_raw
)

query_exclusion_qc["n_D3_EC_after_NT_SAFE_exclusion"] = (
    targeting_ec
)

query_exclusion_qc.to_csv(
    QUERY_EXCLUSION_QC_PATH,
    index=False,
)

print(
    "Saved query-guide exclusion QC:",
    QUERY_EXCLUSION_QC_PATH,
)

print(
    "Total query-guide carrier assignments removed because of "
    "NT/SAFE co-carriage:",
    int(
        query_exclusion_qc[
            "n_D3_query_carriers_removed_for_NT_SAFE_cocarriage"
        ].sum()
    ),
)


# ------------------------------------------------------------------
# Control cells: exclude any D3 cell carrying any Hippo-targeting guide.
# ------------------------------------------------------------------

hippo_carrier_mask_d3 = row_any(
    guide_assignment_d3,
    hippo_guide_cols,
)

eligible_control_cell_mask = ~hippo_carrier_mask_d3


# ------------------------------------------------------------------
# Individual NT/SAFE-guide rows
# ------------------------------------------------------------------

control_matrix = guide_assignment_d3[:, control_cols]

individual_control_total = column_sums(
    control_matrix,
    row_mask=eligible_control_cell_mask,
)

individual_control_ec = column_sums(
    control_matrix,
    row_mask=eligible_control_cell_mask & d3_ec_mask,
)

individual_control_nonec = (
    individual_control_total - individual_control_ec
)

individual_control_pct = safe_percent(
    individual_control_ec,
    individual_control_total,
)

individual_control_out = guide_map.loc[
    control_cols,
    ["guide_id", "guide_name", "guide_index"],
].copy()

individual_control_out.insert(
    0,
    "row_type",
    "individual_NT_guide_excl_Hippo",
)

individual_control_out["target"] = "individual_NT_guide"
individual_control_out["control_definition"] = (
    "individual NT/SAFE guide, excluding Hippo carriers"
)
individual_control_out["exclusion_rule"] = "exclude_Hippo_only"
individual_control_out["n_D3_cells"] = individual_control_total
individual_control_out["n_D3_EC_cells"] = individual_control_ec
individual_control_out["n_D3_nonEC_cells"] = individual_control_nonec
individual_control_out["pct_EC_D3"] = individual_control_pct


# ------------------------------------------------------------------
# Pooled NT/SAFE control group
#
# A cell is in the pooled control if it carries >=1 control guide
# and carries no Hippo guide.
# ------------------------------------------------------------------

pooled_control_mask_d3 = (
    raw_control_carrier_mask_d3
    & eligible_control_cell_mask
)

pooled_control_total = int(
    pooled_control_mask_d3.sum()
)

pooled_control_ec = int(
    (pooled_control_mask_d3 & d3_ec_mask).sum()
)

pooled_control_nonec = (
    pooled_control_total - pooled_control_ec
)

pooled_control_pct = (
    np.nan
    if pooled_control_total == 0
    else 100.0 * pooled_control_ec / pooled_control_total
)

pooled_control_out = pd.DataFrame([{
    "row_type": "control_group",
    "guide_id": pd.NA,
    "guide_name": pd.NA,
    "guide_index": np.nan,
    "target": "NT excluding Hippo",
    "control_definition": (
        "all NT/SAFE-guide cells, excluding Hippo carriers"
    ),
    "exclusion_rule": "exclude_Hippo_only",
    "n_D3_cells": pooled_control_total,
    "n_D3_EC_cells": pooled_control_ec,
    "n_D3_nonEC_cells": pooled_control_nonec,
    "pct_EC_D3": pooled_control_pct,
}])


# ------------------------------------------------------------------
# Combine in the same row order as the prior file:
# targeting guides by guide index,
# individual controls by guide index,
# pooled control group last.
# ------------------------------------------------------------------

output_columns = [
    "row_type",
    "guide_id",
    "guide_name",
    "guide_index",
    "target",
    "control_definition",
    "exclusion_rule",
    "n_D3_cells",
    "n_D3_EC_cells",
    "n_D3_nonEC_cells",
    "pct_EC_D3",
]

targeting_out = targeting_out.sort_values(
    "guide_index"
).reset_index(drop=True)

individual_control_out = individual_control_out.sort_values(
    "guide_index"
).reset_index(drop=True)

output_df = pd.concat(
    [
        targeting_out[output_columns],
        individual_control_out[output_columns],
        pooled_control_out[output_columns],
    ],
    ignore_index=True,
)

# guide_index becomes float because the pooled control row is NA.
output_df["guide_index"] = pd.to_numeric(
    output_df["guide_index"],
    errors="coerce",
).astype(float)

# Count columns should be integer.
for col in [
    "n_D3_cells",
    "n_D3_EC_cells",
    "n_D3_nonEC_cells",
]:
    output_df[col] = output_df[col].astype(int)

output_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved:", OUTPUT_PATH)
print("Output shape:", output_df.shape)
print("\nrow_type counts:")
print(output_df["row_type"].value_counts())

print("\nPooled control:")
display(output_df.tail(1))

output_df.head()


## 6. Final QC

In [ ]:

# ============================================================
# FINAL QC
# ============================================================

assert output_df.columns.tolist() == [
    "row_type",
    "guide_id",
    "guide_name",
    "guide_index",
    "target",
    "control_definition",
    "exclusion_rule",
    "n_D3_cells",
    "n_D3_EC_cells",
    "n_D3_nonEC_cells",
    "pct_EC_D3",
]

assert (
    output_df["n_D3_EC_cells"]
    + output_df["n_D3_nonEC_cells"]
    == output_df["n_D3_cells"]
).all()

# Query-guide rows must exclude cells carrying any NT/SAFE guide.
assert (
    output_df.loc[
        output_df["row_type"].eq("targeting_guide"),
        "exclusion_rule",
    ]
    .eq("exclude_NT_SAFE_cocarriers")
    .all()
)

# Control rows retain the prior Hippo exclusion.
assert (
    output_df.loc[
        output_df["row_type"].ne("targeting_guide"),
        "exclusion_rule",
    ]
    .eq("exclude_Hippo_only")
    .all()
)

# Counts after exclusion cannot exceed raw carrier counts.
assert (
    query_exclusion_qc[
        "n_D3_query_carriers_after_NT_SAFE_exclusion"
    ]
    <= query_exclusion_qc[
        "n_D3_query_carriers_before_NT_SAFE_exclusion"
    ]
).all()

# The eligible query population has no NT/SAFE-guide carriers.
assert not np.any(
    raw_control_carrier_mask_d3[eligible_query_cell_mask]
)

print("QC passed.")
print(
    "Query-guide carriers and NT/SAFE-guide carriers are mutually exclusive "
    "in the targeting-guide rows."
)

removed_total = int(
    query_exclusion_qc[
        "n_D3_query_carriers_removed_for_NT_SAFE_cocarriage"
    ].sum()
)

# Useful summary for comparison with the prior export.
qc_summary = pd.DataFrame({
    "metric": [
        "total output rows",
        "targeting guide rows",
        "individual NT guide rows",
        "pooled control rows",
        "query-guide carrier assignments removed for NT/SAFE co-carriage",
        "pooled control D3 cells",
        "pooled control D3 EC cells",
        "pooled control D3 non-EC cells",
        "pooled control EC percent",
    ],
    "value": [
        len(output_df),
        int(output_df["row_type"].eq("targeting_guide").sum()),
        int(output_df["row_type"].eq("individual_NT_guide_excl_Hippo").sum()),
        int(output_df["row_type"].eq("control_group").sum()),
        removed_total,
        pooled_control_total,
        pooled_control_ec,
        pooled_control_nonec,
        pooled_control_pct,
    ],
})

display(qc_summary)

# Close the backed H5AD handle.
if USE_BACKED_MODE:
    try:
        adata_int.file.close()
    except Exception:
        pass
